# Statement Forecast v2: walk-forward naive / pooled-AR combination

Independent research lab. Not affiliated with any employer or financial institution. Not investment advice.

Pesaran, M. H., Pick, A., & Timmermann, A. (2026). *Forecasting with panel data: Estimation uncertainty versus
parameter heterogeneity.* Quantitative Economics 17(2), 342–393. [doi:10.3982/QE2589](https://doi.org/10.3982/QE2589)
(CEPR DP17123, 2022). Supporting: Wang, Hyndman, Li & Kang (2023), *Forecast combinations: An over 50-year review*,
IJF 39(4), 1518–1547, [doi:10.1016/j.ijforecast.2022.11.005](https://doi.org/10.1016/j.ijforecast.2022.11.005).

The fixed specification and pass bar are loaded from `test_design_combo.json`. For each line and origin t, the combination
is C = N + w·(P − N) with N = naive and P = the shared pooled AR on log growth (both imported from `y9c.forecast`).
The weight w is the grid argmin of summed absolute error over pseudo-out-of-sample cases with target ≤ t, computed from the
panel truncated to quarters ≤ t. Same lines, panel, cases and window as v1. No tuning, trimming or outlier removal.

The real-data run refuses to start unless the working tree is clean and `test_design_combo.json` hashes to the
pre-registration pinned in `statement_forecast.prereg`. The real data are run once and published whatever the verdict.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess
from datetime import datetime, timezone
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tracks/y9c-panel/core_items.yaml').exists())
for track in ['tracks/y9c-panel', 'tracks/statement-forecast']:
    sys.path.insert(0, str(ROOT / track))
from y9c.panel import PROCESSED
from statement_forecast.combo import COMBO_RESULTS, load_combo_design, run_all, write_outputs
from statement_forecast.prereg import verify_combo_preregistration

run_start = datetime.now(timezone.utc).isoformat()
def git(*args):
    return subprocess.check_output(['git', *args], cwd=ROOT)
# Pinned pre-registration: clean tree and the committed design digest, or the run stops here.
prereg = verify_combo_preregistration()
digest = prereg['test_design_sha256']
print('Verified test_design_combo.json SHA-256 against pinned pre-registration:', digest)
run_meta = dict(run_start_time_utc=run_start, git_head=git('rev-parse', 'HEAD').decode().strip(),
                prereg_commit=prereg['prereg_commit'], prereg_commit_time=prereg['prereg_commit_time'],
                test_design_sha256=digest)
design = load_combo_design()
display(Markdown(design['research_only']))
display(Markdown('**Question:** ' + design['question']))
display(pd.DataFrame(design['lines']))
display(Markdown('**Pass bar:** ' + design['pass_bar_verbatim']))
print(json.dumps({key: design[key] for key in ['test_window', 'universe', 'evaluation_cases', 'method']},
                 indent=2, ensure_ascii=False))

# Smoke runs set SF_PANEL_PATH (synthetic panel) and SF_RESULTS_DIR; the real run uses the shared cache.
real_run = 'SF_PANEL_PATH' not in os.environ
panel_path = PROCESSED / 'panel_wide.parquet' if real_run else Path(os.environ['SF_PANEL_PATH'])
results_dir = Path(os.environ.get('SF_RESULTS_DIR', str(COMBO_RESULTS)))
vintage_path = Path(os.environ.get('SF_VINTAGE_PATH', str(panel_path.parent / 'vintage.json')))
expected_cases = design['test_window']['expected_cases_per_line'] if real_run else None
panel = pd.read_parquet(panel_path)
print('Real run:', real_run, '| panel rows:', len(panel),
      '| quarters:', panel.report_date.min().date(), 'to', panel.report_date.max().date())

## Fixed out-of-sample evaluation

Evaluation cases, naive, seasonal naive and pooled AR come from `y9c.forecast.baseline_forecasts` exactly as in v1.
The combination uses the origin-t weight. MAE/RMSE are thousands USD, equally weighted by BHC-quarter.
Positive DM t favors the combination; two-sided p uses target-quarter CR1 clusters and Student t(G−1).
Comparisons with pooled AR and seasonal naive, RMSE, squared-error DM, per-year results, the weight path and fallback
counts are **DESCRIPTIVE ONLY**; there is no multiplicity adjustment.

In [ ]:
results = run_all(panel, design, expected_cases=expected_cases)
print(results['headline'])
error_format = {'mae': '{:,.0f}', 'rmse': '{:,.0f}'}
for line in design['lines']:
    key = line['key']
    display(Markdown('### ' + line['label']))
    display(results['overall_errors'].query('line == @key').style.format(error_format))
    dm = results['dm_tests'].query('line == @key')
    display(dm.loc[dm.scope.eq('full')].style.format({'t_stat': '{:.3f}', 'p_value': '{:.4f}'}))
    print(json.dumps(results['verdicts'][key], indent=2))
    print('PER-YEAR ERRORS AND DM: DESCRIPTIVE ONLY')
    display(results['by_year_errors'].query('line == @key').style.format(error_format))
    display(dm.loc[dm.scope.ne('full')].style.format({'t_stat': '{:.3f}', 'p_value': '{:.4f}'}))
    audit = results['audits'][key]
    print('Cases:', audit['n_forecasts_per_method'], '| pooled AR fallbacks:', audit['ar_fallbacks'],
          '| min-quarter weight fallbacks:', audit['weight_min_quarters_fallbacks'])
print('WEIGHT PATH: DESCRIPTIVE ONLY')
display(results['weights'].pivot(index='origin', columns='line', values='w'))
write_outputs(results, results_dir, vintage_path, run_meta)

## Aggregate figures

Navy palette; MAE in whole thousands USD. The weight path is descriptive only. No per-bank forecast rows are exported.

In [ ]:
COLORS = ['#0a2540', '#0a4080', '#5a7a9a', '#8daabe']
LABELS = {'combo': 'Combination', 'naive': 'Naive', 'seasonal_naive': 'Seasonal naive', 'pooled_ar': 'Pooled panel AR'}
plt.rcParams.update({'axes.prop_cycle': plt.cycler(color=COLORS), 'figure.dpi': 130,
                     'axes.labelcolor': '#0a2540', 'text.color': '#0a2540',
                     'xtick.color': '#0a2540', 'ytick.color': '#0a2540', 'axes.edgecolor': '#5a7a9a',
                     'grid.color': '#8daabe', 'legend.fancybox': False,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white'})
figures = results_dir / 'figures'
figures.mkdir(parents=True, exist_ok=True)
def save(fig, filename):
    fig.tight_layout()
    fig.savefig(figures / filename, dpi=130, bbox_inches='tight', metadata={'Software': None})
    plt.show()
    plt.close(fig)

mae = results['overall_errors'].pivot(index='line', columns='method', values='mae')
mae = mae.reindex(index=[line['key'] for line in design['lines']], columns=list(LABELS))
mae.index = [line['label'] for line in design['lines']]
fig, ax = plt.subplots(figsize=(8, 5))
mae.rename(columns=LABELS).plot.bar(ax=ax, color=COLORS, rot=0)
ax.set(xlabel='Statement line', ylabel='MAE (thousands USD)', title='One-quarter-ahead forecast MAE')
ax.yaxis.set_major_formatter(FuncFormatter(lambda value, pos: f'{value:,.0f}'))
ax.legend(title='Method', loc='upper center', bbox_to_anchor=(.5, -.15), ncol=2, borderaxespad=0)
save(fig, 'mae_by_line.png')

fig, ax = plt.subplots(figsize=(8, 4.5))
for color, line in zip(COLORS, design['lines']):
    w = results['weights'].loc[results['weights'].line.eq(line['key'])]
    dates = pd.PeriodIndex(w.origin, freq='Q').to_timestamp(how='end').normalize()
    ax.step(dates, w.w, where='post', label=line['label'], color=color)
ax.set(xlabel='Origin quarter', ylabel='Weight on pooled AR (w)', ylim=(-.02, 1.02),
       title='Walk-forward combination weight — descriptive only')
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.18), ncol=3, borderaxespad=0)
save(fig, 'weight_path.png')

## Interpretation and limitations

The headline reports all three pre-registered verdicts, including failures. MAE must strictly improve on naive
(unrounded) and the full-sample absolute-error DM p must be ≤ 0.05; an undefined p fails. Everything else is descriptive.

Selection disclosure: pooled AR was chosen as the second component after seeing v1's descriptive tables on this same
2022Q1–2026Q2 window, and this is the second method scored on that window, so a pass is weaker evidence than a pass on
fresh data. This is a report-quarter backtest on currently downloaded filings; filing delays and revisions are not
modeled. RSSD histories have no pro-forma merger adjustments. Research only; not investment advice.